# 008 — Homoglyph-Augmented LoRA Training

Replicates experiment 006 (Qwen2.5-1.5B LoRA) with one change: **homoglyph data augmentation** on 10% of AI-labeled training samples.

Based on [mdok (Macko et al., 2025)](https://github.com/kinit-sk/mdok):
- 5% per-character chance of unicode homoglyph swap
- 5% per-character chance of zero-width joiner insertion

**Requirements**: GPU runtime (T4 or better).

In [1]:
# =====================================================
# CELL 0 — Environment config
# =====================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"
os.environ["DATASETS_DISABLE_PROGRESS"] = "1"
print("Environment configured.")

Environment configured.


In [2]:
# =====================================================
# CELL 1 — Install dependencies
# =====================================================
%pip install -q peft gdown bitsandbytes confusables
print("Dependencies installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 37.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.5/268.5 kB 27.7 MB/s eta 0:00:00
Dependencies installed.


In [3]:
# =====================================================
# CELL 2 — Imports & device check
# =====================================================
import json
import numpy as np
import pandas as pd
import torch
import gdown
import random
from datasets import Dataset, disable_progress_bar
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer,
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import roc_auc_score, f1_score
from confusables import confusable_characters

disable_progress_bar()

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Constants ──────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-1.5B"
MAX_LENGTH = 512

# ── LoRA hyper-parameters (same as 006) ────────────
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["q_proj", "v_proj"]

# ── Homoglyph augmentation parameters (from mdok) ──
H_PROBABILITY = 0.05    # per-char chance of unicode swap
ZWJ_PROBABILITY = 0.05  # per-char chance of ZWJ insertion
AUG_FRACTION = 0.10     # fraction of AI samples to augment

Using device: cuda
GPU: NVIDIA L4


In [4]:
# =====================================================
# CELL 3 — Download data from Google Drive
# =====================================================
train_id = '1k84YY0p8JTuTE40yNEkgzsdaSxt_1nl-'
val_id   = '12szC1TcNPN9KULPNjTZsEyG8RafxfsJy'

BASE = "/content/data/"
os.makedirs(BASE, exist_ok=True)

TRAIN_FILE = BASE + "train.jsonl"
VAL_FILE   = BASE + "val.jsonl"

print("Downloading train...")
gdown.download(f'https://drive.google.com/uc?id={train_id}', TRAIN_FILE, quiet=True)
print("Downloading val...")
gdown.download(f'https://drive.google.com/uc?id={val_id}', VAL_FILE, quiet=True)
print("Downloads complete.")

Downloads complete.


In [5]:
# =====================================================
# CELL 4 — Load data & apply homoglyph augmentation
# =====================================================
def load_jsonl(path):
    with open(path, 'r') as f:
        return pd.DataFrame([json.loads(l) for l in f])


def homoglyph(input_text):
    """Replace characters with unicode look-alikes & insert ZWJ.
    Matches mdok: H_PROBABILITY=0.05, ZWJ_PROBABILITY=0.05."""
    output = ""
    for char in input_text:
        if random.random() < H_PROBABILITY:
            candidates = confusable_characters(char)
            output += candidates[int(random.random() * len(candidates))]
        else:
            output += char
        if random.random() < ZWJ_PROBABILITY:
            output += "\u200D"
    return output


train_df = load_jsonl(TRAIN_FILE)
val_df   = load_jsonl(VAL_FILE)
print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}")
print(f"Label distribution (train): {dict(train_df['label'].value_counts())}")

# ── Apply homoglyph augmentation ──────────────────
# 10% of AI-labeled (label=1) samples get augmented
n_before = len(train_df)
aug_count = 0
augmented_texts = []
for text, label in zip(train_df['text'], train_df['label']):
    if (label == 1 or str(label) == 'machine') and random.random() < AUG_FRACTION:
        augmented_texts.append(homoglyph(text))
        aug_count += 1
    else:
        augmented_texts.append(text)

train_df['text'] = augmented_texts
print(f"\nHomoglyph augmentation applied to {aug_count:,} / {n_before:,} samples ({aug_count/n_before:.1%})")

# Show example
ai_idx = train_df[train_df['label'] == 1].index[0]
sample = train_df.loc[ai_idx, 'text'][:200]
print(f"\nSample augmented text (first 200 chars):\n{repr(sample)}")

Train: 23,707  |  Val: 3,589
Label distribution (train): {1: np.int64(14606), 0: np.int64(9101)}

Homoglyph augmentation applied to 1,446 / 23,707 samples (6.1%)

Sample augmented text (first 200 chars):
'Duke Ellington, a titan of jazz, revolutionized the genre through his innovative compositions, showcasing a remarkable ability to integrate voice and instrumental music. Among the notable figures who '


In [6]:
# =====================================================
# CELL 5 — Tokenize datasets
# =====================================================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
print(f"Tokenizer loaded. Pad token: {tokenizer.pad_token!r}, padding side: {tokenizer.padding_side}")

def tokenize_df(df):
    ds = Dataset.from_pandas(df[['text', 'label']])
    def tok(ex):
        return tokenizer(
            ex['text'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )
    return (
        ds.map(tok, batched=True, batch_size=1000, num_proc=1, remove_columns=['text'])
          .rename_column('label', 'labels')
    )

print("Tokenizing train...")
train_dataset = tokenize_df(train_df)
print("Tokenizing val...")
val_dataset   = tokenize_df(val_df)
print("Done.")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Tokenizer loaded. Pad token: '<|endoftext|>', padding side: left
Tokenizing train...
Tokenizing val...
Done.


In [7]:
# =====================================================
# CELL 6 — Load base model + wrap with LoRA
# =====================================================
print("Loading base model...")

config = AutoConfig.from_pretrained(MODEL_NAME)
config.pad_token_id = config.eos_token_id
config.num_labels = 2
config.problem_type = "single_label_classification"

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    torch_dtype=torch.bfloat16,
)

base_model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

print(f"Base model parameters: {base_model.num_parameters():,}")

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading base model...


Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-1.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Base model parameters: 1,543,717,376
trainable params: 2,182,144 || all params: 1,545,899,520 || trainable%: 0.1412


In [8]:
# =====================================================
# CELL 7 — Training setup
# =====================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
    preds = np.argmax(logits, axis=-1)
    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except Exception:
        auc = 0.5
    return {
        'roc_auc': auc,
        'f1': f1_score(labels, preds),
        'accuracy': (preds == labels).mean(),
    }

use_cuda = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="/content/lora_checkpoints",
    use_cpu=not use_cuda,
    bf16=use_cuda,
    per_device_train_batch_size=8 if use_cuda else 2,
    per_device_eval_batch_size=16 if use_cuda else 4,
    gradient_accumulation_steps=2 if use_cuda else 8,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=100,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=200,
    logging_steps=50,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True,
    report_to="none",
    disable_tqdm=True,
    dataloader_num_workers=0,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)
print("Trainer ready.")

Trainer ready.


In [9]:
# =====================================================
# CELL 8 — Train!
# =====================================================
print("Starting LoRA fine-tuning with homoglyph augmentation...")
trainer.train()
print("Training complete.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting LoRA fine-tuning with homoglyph augmentation...
{'loss': '2.891', 'grad_norm': '65.74', 'learning_rate': '9.8e-05', 'epoch': '0.03374'}
{'loss': '0.7507', 'grad_norm': '13.35', 'learning_rate': '0.000198', 'epoch': '0.06748'}
{'loss': '0.2816', 'grad_norm': '0.6687', 'learning_rate': '0.0001977', 'epoch': '0.1012'}
{'loss': '0.2332', 'grad_norm': '0.03329', 'learning_rate': '0.0001954', 'epoch': '0.135'}
{'eval_loss': '0.0854', 'eval_roc_auc': '0.9961', 'eval_f1': '0.9875', 'eval_accuracy': '0.9841', 'eval_runtime': '150.8', 'eval_samples_per_second': '23.81', 'eval_steps_per_second': '1.493', 'epoch': '0.135'}
{'loss': '0.08898', 'grad_norm': '0.1612', 'learning_rate': '0.0001931', 'epoch': '0.1687'}
{'loss': '0.1101', 'grad_norm': '0.001483', 'learning_rate': '0.0001908', 'epoch': '0.2024'}
{'loss': '0.2941', 'grad_norm': '0.09275', 'learning_rate': '0.0001885', 'epoch': '0.2362'}
{'loss': '0.06968', 'grad_norm': '0.0132', 'learning_rate': '0.0001862', 'epoch': '0.2699'}
{'e

In [10]:
# =====================================================
# CELL 9 — Final evaluation on validation set
# =====================================================
results = trainer.evaluate()
print("Validation results:")
for k, v in results.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

{'eval_loss': '0.01578', 'eval_roc_auc': '0.9999', 'eval_f1': '0.9981', 'eval_accuracy': '0.9975', 'eval_runtime': '150.7', 'eval_samples_per_second': '23.82', 'eval_steps_per_second': '1.493', 'epoch': '3'}
Validation results:
  eval_loss: 0.0158
  eval_roc_auc: 0.9999
  eval_f1: 0.9981
  eval_accuracy: 0.9975
  eval_runtime: 150.6801
  eval_samples_per_second: 23.8190
  eval_steps_per_second: 1.4930
  epoch: 3.0000


In [11]:
# =====================================================
# CELL 10 — Save adapter & push to HuggingFace
# =====================================================
from huggingface_hub import HfApi, login

# ── CONFIGURE THESE ────────────────────────────────
HF_TOKEN    = "hf_xxx" 
HF_USERNAME = "hersheys-baklava"
REPO_NAME   = "pan2026-qwen-homoglyph"
# ──────────────────────────────────────────────────

login(token=HF_TOKEN)

ADAPTER_DIR = "/content/lora_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

with open(f"{ADAPTER_DIR}/training_metrics.json", 'w') as f:
    json.dump(results, f, indent=2)

lora_summary = {
    "base_model": MODEL_NAME,
    "experiment": "008-homoglyph",
    "augmentation": {
        "type": "homoglyph",
        "h_probability": H_PROBABILITY,
        "zwj_probability": ZWJ_PROBABILITY,
        "aug_fraction": AUG_FRACTION,
    },
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "target_modules": LORA_TARGET_MODULES,
    "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
    "total_params": sum(p.numel() for p in model.parameters()),
    "max_length": MAX_LENGTH,
    "num_epochs": 3,
    "learning_rate": 2e-4,
}
with open(f"{ADAPTER_DIR}/lora_config_summary.json", 'w') as f:
    json.dump(lora_summary, f, indent=2)

print(f"Adapter saved locally to {ADAPTER_DIR}")
print(f"Adapter size: {sum(os.path.getsize(os.path.join(ADAPTER_DIR, f)) for f in os.listdir(ADAPTER_DIR)) / 1024:.1f} KB")

Adapter saved locally to /content/lora_adapter
Adapter size: 19696.8 KB


In [12]:
# =====================================================
# CELL 11 — Upload adapter to HuggingFace Hub
# =====================================================
api = HfApi()
repo_id = f"{HF_USERNAME}/{REPO_NAME}"

api.create_repo(repo_id=repo_id, exist_ok=True, private=True)

api.upload_folder(
    folder_path=ADAPTER_DIR,
    repo_id=repo_id,
    commit_message="Upload LoRA adapter (homoglyph-augmented) for Qwen2.5-1.5B",
)

print(f"\n=== Adapter uploaded to: https://huggingface.co/{repo_id} ===")
print(f"\nTo load for evaluation:")
print(f"  base = AutoModelForSequenceClassification.from_pretrained('{MODEL_NAME}', config=config)")
print(f"  model = PeftModel.from_pretrained(base, '{repo_id}')")
print(f"  model = model.merge_and_unload()")


=== Adapter uploaded to: https://huggingface.co/hersheys-baklava/pan2026-qwen-homoglyph ===

To load for evaluation:
  base = AutoModelForSequenceClassification.from_pretrained('Qwen/Qwen2.5-1.5B', config=config)
  model = PeftModel.from_pretrained(base, 'hersheys-baklava/pan2026-qwen-homoglyph')
  model = model.merge_and_unload()
